# رابط ساده بخش داده — نسخه 0.6.0a2

تمام قیمت‌های این دفترچه مصنوعی‌اند. هیچ شبکه واقعی و هیچ تحلیل یا مدل هوش مصنوعی اجرا نمی‌شود.

### سلول ۱ — ورود کتابخانه‌ها

In [ ]:
from pathlib import Path
from importlib.util import find_spec
from unittest.mock import patch
import pandas as pd
import httpx
import pydara
from pydara import tse
from pydara.tse import FetchPolicy
from pydara.tse.samples import sample_client
print(pydara.__version__)

### سلول ۲ — تابع مقایسه فایل و جلوگیری از شبکه

In [ ]:
def check_roundtrip(frame, path):
    receipt = tse.save(frame, path, overwrite=True)
    restored = tse.load(path)
    pd.testing.assert_frame_equal(frame, restored, check_dtype=True)
    return {"file": str(path), "state": receipt.state, "rows": len(restored)}

def block_network(*args, **kwargs):
    raise AssertionError("Unexpected HTTP during offline test")

### سلول ۳ — تنظیمات

In [ ]:
OUTPUT_DIR = Path("dara_review_sample_output")
SYMBOLS = pd.Series(["فولاد", "شستا", "فولاد"], dtype="string")
CACHE_DIR = OUTPUT_DIR / "synthetic_cache"

### سلول ۴ — داده مصنوعی و نمایش خوانا

In [ ]:
history = tse.load_sample("daily_prices_v1")
display(tse.select_columns(history, "compact"))
display(tse.schema_info("daily_bars"))
display(tse.validate(history, schema="daily_bars"))

### سلول ۵ — درخواست گروهی با Series و جدول تک‌نماد

In [ ]:
with sample_client() as client:
    quote_frame = client.get_quote_frame("فولاد")
    quotes = client.get_quotes(SYMBOLS, on_error="collect")
    batch = client.get_history_many(SYMBOLS, on_error="collect")
display(tse.select_columns(quote_frame))
display(tse.select_columns(quotes))
display(batch.requests)
display(batch.errors)

### سلول ۶ — رفت‌وبرگشت قالب‌ها

In [ ]:
results = []
for extension, module in [("csv", None), ("xlsx", "openpyxl"), ("sqlite", None), ("parquet", "pyarrow")]:
    if module is not None and find_spec(module) is None:
        results.append({"file": extension, "state": "SKIPPED: optional dependency unavailable"})
        continue
    results.append(check_roundtrip(history, OUTPUT_DIR / ("sample." + extension)))
display(pd.DataFrame(results))

### سلول ۷ — ذخیره و خواندن آفلاین صریح

In [ ]:
with sample_client(storage=True, data_dir=CACHE_DIR) as client:
    quote = client.get_quote("فولاد")
    client.flush()

with patch.object(httpx.Client, "send", new=block_network):
    with sample_client(storage=True, data_dir=CACHE_DIR) as client:
        cached = client.get_quote_frame("فولاد", policy=FetchPolicy.offline(allow_stale=True))
        assert client.request_count == 0
        display(tse.select_columns(cached))

### سلول ۸ — مرجع واقعی رابط

In [ ]:
display(tse.api_reference("get_history"))
print(tse.frame_metadata(history))